In [ ]:
from dotenv import load_dotenv, find_dotenv
from pprint import pprint

True

In [ ]:

load_dotenv(find_dotenv(), override=True)

In [ ]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [ ]:
# give your tool a custom name and description by explicitly specifying it
# if you do not specify a custom name, the tool will use the function name as its name
@tool("square_root", description="Calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5

# Entering doc string is important if you are not providing a custom name and description for the tool
# The doc string will be used as the tool's description in that case

In [14]:
# check the type of the tool
print(type(square_root))


<class 'langchain_core.tools.structured.StructuredTool'>


In [15]:
print(square_root.args)

{'x': {'title': 'X', 'type': 'number'}}


Now that we know how arguments are to be passes, we can invoke the tool in the following ways:

In [9]:
# invoke the tool by the given name 
square_root.invoke({"x": 16})

4.0

In [10]:
# or by the implicit name (function nanem)
tool1.invoke({"x": 64})

8.0

In [16]:
import langchain
print(langchain.__version__)

1.5.1


In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate the square root and square of any number."
)

In [19]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

The square root of 467 is approximately 21.61018278497431. If you’d like it rounded to a specific number of decimals, tell me how many.


In [ ]:
pprint(response['messages'])

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='69c74539-4c7b-428f-8db6-f5faf6c1ece1'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1756, 'prompt_tokens': 158, 'total_tokens': 1914, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1728, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ElyuAB6V6ievqiFWoJnTPu8WxLr3s', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f48c-f2d2-76fa-a2a0-9bf1f6e3f3b1-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_XzQ0EeZui4xHKpPKEO4DYbrl', 'type': 'tool_call'}

In [21]:
print(response["messages"][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_XzQ0EeZui4xHKpPKEO4DYbrl', 'type': 'tool_call'}]


<small>

```
.tool_calls 
```

Is a list containing the tool invocations the model requested. It tells which tool the LLM selected, what arguments it passed and the internal tool call ID.

</small>

In [29]:
print(response["messages"][2].content)

21.61018278497431


In [30]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

My training data goes up to June 2024. I don’t have real-time web access, so I can’t pull in events or data after that unless you provide it in the chat. I can help reason about newer topics if you give me context or sources, but you should verify time-sensitive facts with up-to-date references. And if you need arithmetic help (like squares or square roots), I’m ready to compute those.


# Web Search

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
import os
from pprint import pprint

# Tavily's search client for performing web searches
# You can run this without setting the API key but that has a very small anonymous quota associated with your account
# So it is better to use an API key for higher quota and reliability
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': '| Image 5 Seal of San Francisco |\n| Image 6 Flag of San Francisco |\n| Image 7.jpg) Incumbent Daniel Lurie since January 8, 2025 |\n| Government of San Francisco |\n| Style "Style (form of address)") | His Honor Mr. Mayor |\n| Residence | No official residence |\n| Term length |  Four years, renewable once |\n| Inaugural holder | John W. Geary |\n| Formation | 1850 |\n| Salary | $1( |\n| Website | Office of the Mayor | [...] There have been 45 individuals who have served as mayor in San Francisco since 1850, when California became a state following the American Conquest of California. Prior to the conquest, Californios served as Mayor of San Francisco during the Spanish and Mexican eras since 1779.\n\nThe current mayor is Democra

<small>

```python
def web_search(query: str) -> Dict[str, Any]:
```

The function definition means the input will be string and the output will be a dictionary which will have string keys and can contain values of any type (Any).

The function could also simply be written as the following: 

```python
def web_search(query):
    return tavily_client.search(query)
```

The previous way just gives type hints and provides extra information.

</small>

In [32]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

In [50]:
pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='e8f4da02-640a-4cd5-a1de-a9b6032da600'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 350, 'prompt_tokens': 133, 'total_tokens': 483, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Elz74LI1PyWjPQNJfTMbqmFRLUGjW', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a1f499-2acf-70a7-8931-46869e0a183b-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco 2026'}, 'id': 'call_y8

In [ ]:
pprint(response['messages'][-1].content)

('Daniel Lurie is the current mayor of San Francisco. He is the 46th mayor and '
 'took office on January 8, 2025, succeeding London Breed.')


In [51]:
pprint(response["messages"][1].tool_calls)

[{'args': {'query': 'current mayor of San Francisco 2026'},
  'id': 'call_y8Ul5DvSmYBk24vE5TnzLtuO',
  'name': 'web_search',
  'type': 'tool_call'}]


To see all tools called with their name, arguments and ID, do the following:

In [48]:
for msg in response["messages"]:
    if hasattr(msg, "tool_calls"):
        for tc in msg.tool_calls:
            print("Tool:", tc["name"])
            print("Args:", tc["args"])
            print("Tool ID:", tc["id"])
            print()

Tool: web_search
Args: {'query': 'current mayor of San Francisco 2026'}
Tool ID: call_y8Ul5DvSmYBk24vE5TnzLtuO

